In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv("/home/rgukt/Downloads/Titanic-Dataset.csv")

In [4]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [5]:
df = df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1)

In [6]:
df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [7]:
df.shape

(891, 8)

In [8]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [9]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(["Survived"],axis=1),df["Survived"],test_size=0.2,random_state=42)

In [11]:
x_train.shape,x_test.shape

((712, 7), (179, 7))

In [13]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [14]:
#Handling missing values
trf1 = ColumnTransformer(transformers = [
    ("impute_age",SimpleImputer(),[2]),#age-2nd columns(0-based index)
    ("impute_embarked",SimpleImputer(strategy="most_frequent"),[6])#(embarked-6th column)
],remainder = "passthrough")

In [15]:
#One hot encoding(sex,embarked)
trf2 = ColumnTransformer(transformers = [
    ("ohe_sex_embarked",OneHotEncoder(sparse_output= False,handle_unknown="ignore"),[1,6])
],remainder = "passthrough")

In [17]:
#scaling(minmaxscaler - because we are doing feature selection otherwise use standard scaler)
trf3 = ColumnTransformer(transformers = [
         ("scale",MinMaxScaler(),slice(0,10))   
])

In [18]:
#Feature selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
trf4 = SelectKBest(score_func = chi2,k=8)

In [19]:
#Model
trf5 = DecisionTreeClassifier()

In [21]:
#Creating pipeline is different from make pipeline 
pipe = Pipeline([
    ("trf1",trf1),
    ("trf2",trf2),
    ("trf3",trf3),
    ("trf4",trf4),
    ("trf5",trf5)
])

In [22]:
#Alternate make_pipeline(no need of naming)
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [23]:
pipe.fit(x_train,y_train)

,steps,"[('columntransformer-1', ...), ('columntransformer-2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
#Explore pipeline
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x7f8690f9f7e0>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [27]:
pipe.named_steps["columntransformer-1"]

,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [28]:
pipe.named_steps["columntransformer-1"].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 5])]

In [29]:
pipe.named_steps["columntransformer-1"].transformers_[0]

('impute_age', SimpleImputer(), [2])

In [30]:
pipe.named_steps["columntransformer-1"].transformers_[0][1].statistics_

array([29.49884615])

In [31]:
pipe.named_steps["columntransformer-1"].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [32]:
y_pred = pipe.predict(x_test)

In [33]:
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test,y_pred)

In [34]:
print(score)

0.6256983240223464


In [37]:
#Cross validation using cross_Val_Score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,x_train,y_train,cv=5,scoring = "accuracy").mean()

0.6391214419383433

In [40]:
#Exporting pipeline
import pickle
pickle.dump(pipe,open("titanic_dataset/models/pipeline.pkl","wb"))